In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 第二步：读取四个数据表

In [28]:
import pandas as pd

# 读取四个表（header=1 很关键！）
basic_school = pd.read_excel("Basic School Stats.csv", header=1)
basic_opponent = pd.read_excel("Basic Opponent Stats.csv", header=1)
advanced_school = pd.read_excel("Advanced School Stats.csv", header=1)
advanced_opponent = pd.read_excel("Advanced Opponent Stats.csv", header=1)

# 看一下数据
print(advanced_school.head())

   Rk             School   G   W   L   W-L%    SRS    SOS  Unnamed: 8  W.1  \
0   1  Abilene Christian  33  14  19  0.424  -7.38  -1.45         NaN    5   
1   2          Air Force  32   3  29  0.094 -13.75   4.60         NaN    0   
2   3         Akron NCAA  35  29   6  0.829   8.48  -2.91         NaN   17   
3   4       Alabama NCAA  35  25  10  0.714  23.03  14.57         NaN   13   
4   5        Alabama A&M  33  18  15  0.545 -13.31 -10.81         NaN   10   

   ...   3PAr    TS%  TRB%  AST%  STL%  BLK%   eFG%  TOV%  ORB%  FT/FGA  
0  ...  0.328  0.529  50.4  54.7  14.0   8.1  0.489  17.7  32.6   0.284  
1  ...  0.418  0.516  44.8  55.9   8.3   6.8  0.489  19.8  21.4   0.225  
2  ...  0.448  0.610  53.3  57.3  10.4   9.1  0.585  13.1  33.9   0.211  
3  ...  0.543  0.590  50.5  54.3   8.6  11.2  0.552  11.2  32.4   0.276  
4  ...  0.343  0.545  49.9  49.4   8.8   7.9  0.496  14.7  28.0   0.331  

[5 rows x 34 columns]


# 第三步：统一列名（非常关键‼️）

In [29]:
def clean_columns(df):
    df.columns = df.columns.str.strip()  # 去空格
    df.columns = df.columns.str.replace('%', '_pct')
    df.columns = df.columns.str.replace('.', '_')
    df.columns = df.columns.str.replace(' ', '_')
    return df

basic_school = clean_columns(basic_school)
basic_opponent = clean_columns(basic_opponent)
advanced_school = clean_columns(advanced_school)
advanced_opponent = clean_columns(advanced_opponent)

# 第四步：只保留你需要的变量

In [30]:
# Advanced School
adv_school_cols = ["School", "ORtg", "eFG_pct", "TOV_pct", "ORB_pct", "FTr"]

# Advanced Opponent
adv_opp_cols = ["School", "ORtg", "eFG_pct", "TOV_pct", "ORB_pct"]
advanced_opponent = advanced_opponent.rename(columns={
    "ORtg": "Opp_ORtg",
    "eFG_pct": "Opp_eFG_pct",
    "TOV_pct": "Opp_TOV_pct",
    "ORB_pct": "Opp_ORB_pct"
})

# Basic School（补充变量）
basic_cols = ["School", "STL", "BLK", "AST", "TRB"]

# 保留列
adv_school = advanced_school[adv_school_cols]
adv_opp = advanced_opponent[["School", "Opp_ORtg", "Opp_eFG_pct", "Opp_TOV_pct", "Opp_ORB_pct"]]
basic = basic_school[basic_cols]

# 第五步：合并四个表

In [31]:
# 合并 Advanced School + Advanced Opponent
df = pd.merge(adv_school, adv_opp, on="School", how="inner")

# 再合并 Basic
df = pd.merge(df, basic, on="School", how="inner")

print(df.head())
print("数据维度:", df.shape)

              School   ORtg  eFG_pct  TOV_pct  ORB_pct    FTr  Opp_ORtg  \
0  Abilene Christian  101.1    0.489     17.7     32.6  0.403     104.1   
1          Air Force   91.3    0.489     19.8     21.4  0.351     118.5   
2         Akron NCAA  122.7    0.585     13.1     33.9  0.279     103.3   
3       Alabama NCAA  122.1    0.552     11.2     32.4  0.360     110.7   
4        Alabama A&M  105.9    0.496     14.7     28.0  0.449     106.6   

   Opp_eFG_pct  Opp_TOV_pct  Opp_ORB_pct  STL  BLK  AST   TRB  
0        0.550         21.1         28.2  321   91  440  1047  
1        0.564         13.1         31.3  180   76  389   931  
2        0.504         16.3         29.3  261  109  639  1319  
3        0.492         10.8         32.8  225  174  571  1425  
4        0.504         14.5         28.8  198   93  387  1126  
数据维度: (365, 14)


# 第六步：检查缺失值

In [32]:
print(df.isnull().sum())

# 删除缺失值
df = df.dropna()

print("清理后:", df.shape)

School         0
ORtg           0
eFG_pct        0
TOV_pct        0
ORB_pct        0
FTr            0
Opp_ORtg       0
Opp_eFG_pct    0
Opp_TOV_pct    0
Opp_ORB_pct    0
STL            0
BLK            0
AST            0
TRB            0
dtype: int64
清理后: (365, 14)


# 第七步：定义变量

In [33]:
# 加入目标变量（从 advanced_school 里取）
target_data = advanced_school[["School", "W-L_pct"]]
target_data = target_data.rename(columns={"W-L_pct": "Win_pct"})

# 合并
df = pd.merge(df, target_data, on="School")

# 自变量
features = [
    "ORtg", "eFG_pct", "TOV_pct", "ORB_pct", "FTr",
    "Opp_ORtg", "Opp_eFG_pct", "Opp_TOV_pct", "Opp_ORB_pct",
    "STL", "BLK", "AST", "TRB"
]

# 因变量
y = df["Win_pct"]
X = df[features]

# 第八步：标准化 + 划分数据

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 第九步：建模（线性回归）

In [35]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("R²:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R²: 0.8761035215145052
RMSE: 0.053240250565989244


# 第十步：看变量重要性

In [36]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

coef_df["Abs"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values(by="Abs", ascending=False)

print(coef_df)

        Feature  Coefficient       Abs
0          ORtg     0.080398  0.080398
5      Opp_ORtg    -0.070338  0.070338
12          TRB     0.027352  0.027352
7   Opp_TOV_pct     0.017067  0.017067
6   Opp_eFG_pct    -0.016414  0.016414
1       eFG_pct     0.013243  0.013243
4           FTr     0.012571  0.012571
9           STL    -0.011411  0.011411
2       TOV_pct    -0.011111  0.011111
8   Opp_ORB_pct     0.010995  0.010995
3       ORB_pct    -0.008857  0.008857
11          AST     0.004013  0.004013
10          BLK    -0.001509  0.001509


# 第十一步 做 Lasso，帮助筛选更重要的变量

In [37]:
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

print("Lasso R²:", r2_score(y_test, y_pred_lasso))
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("最佳 alpha:", lasso.alpha_)

Lasso R²: 0.8797882910911057
Lasso RMSE: 0.05244257342982734
最佳 alpha: 0.0012740442704495907


# 第十二步：看 Lasso 选出来的关键变量

In [38]:
lasso_coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lasso.coef_
})

lasso_coef_df["Abs_Coefficient"] = lasso_coef_df["Coefficient"].abs()
lasso_coef_df = lasso_coef_df.sort_values(by="Abs_Coefficient", ascending=False)

print(lasso_coef_df[["Feature", "Coefficient"]])

        Feature  Coefficient
0          ORtg     0.079534
5      Opp_ORtg    -0.067641
6   Opp_eFG_pct    -0.018848
12          TRB     0.015148
4           FTr     0.011622
2       TOV_pct    -0.010564
1       eFG_pct     0.010492
7   Opp_TOV_pct     0.006574
11          AST     0.006066
8   Opp_ORB_pct     0.003786
3       ORB_pct    -0.002334
9           STL     0.000000
10          BLK     0.000000


# 第十三步：再做一个随机森林，看看非线性的重要性

In [39]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest R²:", r2_score(y_test, y_pred_rf))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))

Random Forest R²: 0.8635821498305212
Random Forest RMSE: 0.05586582415233105


In [40]:
rf_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(rf_importance_df)

        Feature  Importance
0          ORtg    0.440983
5      Opp_ORtg    0.347150
12          TRB    0.081744
6   Opp_eFG_pct    0.036595
11          AST    0.018493
2       TOV_pct    0.013234
4           FTr    0.012841
1       eFG_pct    0.009382
10          BLK    0.008629
9           STL    0.008592
3       ORB_pct    0.008389
8   Opp_ORB_pct    0.007446
7   Opp_TOV_pct    0.006519


In [41]:
cv_scores = cross_val_score(
    LinearRegression(),
    scaler.fit_transform(X),
    y,
    cv=5,
    scoring="r2"
)

print("交叉验证 R²:", cv_scores)
print("平均 R²:", cv_scores.mean())

交叉验证 R²: [0.87241617 0.85441486 0.89367327 0.86335746 0.86810004]
平均 R²: 0.8703923617152126


In [42]:
print("=== Linear Regression Coefficients ===")
print(coef_df[["Feature", "Coefficient"]])

print("\n=== Lasso Coefficients ===")
print(lasso_coef_df[["Feature", "Coefficient"]])

print("\n=== Random Forest Importances ===")
print(rf_importance_df)

=== Linear Regression Coefficients ===
        Feature  Coefficient
0          ORtg     0.080398
5      Opp_ORtg    -0.070338
12          TRB     0.027352
7   Opp_TOV_pct     0.017067
6   Opp_eFG_pct    -0.016414
1       eFG_pct     0.013243
4           FTr     0.012571
9           STL    -0.011411
2       TOV_pct    -0.011111
8   Opp_ORB_pct     0.010995
3       ORB_pct    -0.008857
11          AST     0.004013
10          BLK    -0.001509

=== Lasso Coefficients ===
        Feature  Coefficient
0          ORtg     0.079534
5      Opp_ORtg    -0.067641
6   Opp_eFG_pct    -0.018848
12          TRB     0.015148
4           FTr     0.011622
2       TOV_pct    -0.010564
1       eFG_pct     0.010492
7   Opp_TOV_pct     0.006574
11          AST     0.006066
8   Opp_ORB_pct     0.003786
3       ORB_pct    -0.002334
9           STL     0.000000
10          BLK     0.000000

=== Random Forest Importances ===
        Feature  Importance
0          ORtg    0.440983
5      Opp_ORtg    0.347150
12

# 第二模型（具体）

In [43]:
bs = basic_school.copy()
bo = basic_opponent.copy()
adv_s = advanced_school.copy()
adv_o = advanced_opponent.copy()

In [44]:
# ===== Model 2: exactly the 12 variables you want =====

bs = basic_school.copy()
bo = basic_opponent.copy()
adv_s = advanced_school.copy()
adv_o = advanced_opponent.copy()

# 1) Calculate 2PT%
bs["2P_pct"] = (bs["FG"] - bs["3P"]) / (bs["FGA"] - bs["3PA"])
bo["Opp_2P_pct"] = (bo["FG"] - bo["3P"]) / (bo["FGA"] - bo["3PA"])

# 2) Opponent 3PT%
bo["Opp_3P_pct"] = bo["3P_pct"]

# 3) Select exactly the variables you want
bs_use = bs[["School", "3P_pct", "2P_pct"]]

bo_use = bo[["School", "Opp_3P_pct", "Opp_2P_pct"]]

adv_s_use = adv_s[[
    "School", "ORB_pct", "TOV_pct", "STL_pct", "ORtg", "Pace", "SOS"
]]

adv_o_use = adv_o[[
    "School", "Opp_ORtg", "Opp_ORB_pct"
]]

# 4) Merge
df2 = pd.merge(bs_use, bo_use, on="School", how="inner")
df2 = pd.merge(df2, adv_s_use, on="School", how="inner")
df2 = pd.merge(df2, adv_o_use, on="School", how="inner")

# 5) Calculate DRB%
df2["DRB_pct"] = 100 - df2["Opp_ORB_pct"]

# 6) Add target variable
target_data2 = adv_s[["School", "W-L_pct"]].rename(columns={"W-L_pct": "Win_pct"})
df2 = pd.merge(df2, target_data2, on="School", how="inner")

# 7) Your exact feature list
features2 = [
    "3P_pct",
    "2P_pct",
    "Opp_3P_pct",
    "Opp_2P_pct",
    "ORB_pct",
    "DRB_pct",
    "TOV_pct",
    "STL_pct",
    "Pace",
    "SOS"
]

# 8) Build model
data2 = df2[features2 + ["Win_pct"]].dropna()

X2 = data2[features2]
y2 = data2["Win_pct"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

model2 = LinearRegression()
model2.fit(X_train2_scaled, y_train2)

y_pred2 = model2.predict(X_test2_scaled)

print("Model 2 R²:", r2_score(y_test2, y_pred2))
print("Model 2 RMSE:", np.sqrt(mean_squared_error(y_test2, y_pred2)))

coef_df2 = pd.DataFrame({
    "Feature": features2,
    "Coefficient": model2.coef_
})

coef_df2["Abs"] = coef_df2["Coefficient"].abs()
coef_df2 = coef_df2.sort_values(by="Abs", ascending=False)

print(coef_df2)

Model 2 R²: 0.8192538650724647
Model 2 RMSE: 0.06430505573883803
      Feature  Coefficient       Abs
3  Opp_2P_pct    -0.056885  0.056885
6     TOV_pct    -0.043209  0.043209
0      3P_pct     0.042238  0.042238
1      2P_pct     0.041158  0.041158
7     STL_pct     0.035186  0.035186
2  Opp_3P_pct    -0.032474  0.032474
4     ORB_pct     0.030324  0.030324
5     DRB_pct     0.019010  0.019010
8        Pace     0.004854  0.004854
9         SOS     0.002183  0.002183


# 模型三 和疯三做比较

读取 ncaa_result.csv， 统一学校名字

In [45]:
ncaa_result = pd.read_csv("ncaa_result.csv")
print(ncaa_result.head())
print(ncaa_result.columns.tolist())

              School  NCAA_Result
0  Abilene Christian            0
1          Air Force            0
2         Akron NCAA            1
3       Alabama NCAA            3
4        Alabama A&M            0
['School', 'NCAA_Result']


In [46]:
df["School"] = df["School"].str.strip()
ncaa_result["School"] = ncaa_result["School"].str.strip()

# 如果你原数据里还有 NCAA 这个后缀，就去掉
df["School"] = df["School"].str.replace(" NCAA", "", regex=False)
ncaa_result["School"] = ncaa_result["School"].str.replace(" NCAA", "", regex=False)

合并到你的主表，检查有没有没匹配上的球队

In [47]:
df_cls = pd.merge(df, ncaa_result, on="School", how="left")
print(df_cls.head())
print(df_cls[["School", "NCAA_Result"]].head(10))

print(df_cls["NCAA_Result"].isnull().sum())
print(df_cls[df_cls["NCAA_Result"].isnull()][["School"]].head(20))

df2_cls = pd.merge(df2, ncaa_result, on="School", how="left")
print(df2_cls.head())
print(df2_cls[["School", "NCAA_Result"]].head(10))

print(df2_cls["NCAA_Result"].isnull().sum())
print(df2_cls[df2_cls["NCAA_Result"].isnull()][["School"]].head(20))

              School   ORtg  eFG_pct  TOV_pct  ORB_pct    FTr  Opp_ORtg  \
0  Abilene Christian  101.1    0.489     17.7     32.6  0.403     104.1   
1          Air Force   91.3    0.489     19.8     21.4  0.351     118.5   
2         Akron NCAA  122.7    0.585     13.1     33.9  0.279     103.3   
3       Alabama NCAA  122.1    0.552     11.2     32.4  0.360     110.7   
4        Alabama A&M  105.9    0.496     14.7     28.0  0.449     106.6   

   Opp_eFG_pct  Opp_TOV_pct  Opp_ORB_pct  STL  BLK  AST   TRB  Win_pct  \
0        0.550         21.1         28.2  321   91  440  1047    0.424   
1        0.564         13.1         31.3  180   76  389   931    0.094   
2        0.504         16.3         29.3  261  109  639  1319    0.829   
3        0.492         10.8         32.8  225  174  571  1425    0.714   
4        0.504         14.5         28.8  198   93  387  1126    0.545   

   NCAA_Result  
0            0  
1            0  
2            1  
3            3  
4            0  
  

检查类别分布是否合理

In [48]:
print(df_cls["NCAA_Result"].value_counts().sort_index())

NCAA_Result
0    301
1     32
2     16
3      8
4      4
5      2
6      1
7      1
Name: count, dtype: int64


# 开始做模型

模型1

In [49]:
# ===== FINAL CLASSIFICATION MODEL =====

# 1. 定义目标（Sweet 16及以上）
df_cls["Sweet16_or_better"] = (df_cls["NCAA_Result"] >= 3).astype(int)

# 2. 特征（Model 1变量）
features_cls = [
    "ORtg",
    "Opp_ORtg",
    "TRB",
    "eFG_pct",
    "TOV_pct",
    "ORB_pct",
    "Opp_eFG_pct",
    "Opp_TOV_pct",
    "Opp_ORB_pct",
    "FTr",
    "AST"
]

# 3. 构建数据
data_cls = df_cls[["School"] + features_cls + ["Sweet16_or_better"]].dropna()

X_cls = data_cls[features_cls]
y_cls = data_cls["Sweet16_or_better"]
schools = data_cls["School"]

# 4. 划分数据
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, school_train, school_test = train_test_split(
    X_cls, y_cls, schools,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

# 5. 训练模型（重点：class_weight）
from sklearn.ensemble import RandomForestClassifier

rf_cls = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_cls.fit(X_train, y_train)

# ===== 方法一：默认预测 =====
y_pred = rf_cls.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("=== Default Prediction ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))


# ===== 方法二：概率优化（关键🔥）=====
y_prob = rf_cls.predict_proba(X_test)[:, 1]

# 调整阈值（从0.5 → 0.3）
threshold = 0.3
y_pred_adj = (y_prob >= threshold).astype(int)

print("\n=== Adjusted Prediction (threshold=0.3) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_adj))
print(classification_report(y_test, y_pred_adj, zero_division=0))


# ===== 查看最有可能进Sweet16的球队（很加分🔥）=====
result_df = X_test.copy()
result_df["School"] = school_test.values   # ✅ 加回名字
result_df["Actual"] = y_test.values
result_df["Pred_Prob"] = y_prob

print("\n=== Top Predicted Teams ===")
print(result_df.sort_values(by="Pred_Prob", ascending=False)
      [["School", "Pred_Prob", "Actual"]]
      .head(10))


# ===== 特征重要性（一定要写）=====
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": features_cls,
    "Importance": rf_cls.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n=== Feature Importance ===")
print(importance_df)

=== Default Prediction ===
Accuracy: 0.958904109589041
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        70
           1       0.00      0.00      0.00         3

    accuracy                           0.96        73
   macro avg       0.48      0.50      0.49        73
weighted avg       0.92      0.96      0.94        73


=== Adjusted Prediction (threshold=0.3) ===
Accuracy: 0.9863013698630136
              precision    recall  f1-score   support

           0       1.00      0.99      0.99        70
           1       0.75      1.00      0.86         3

    accuracy                           0.99        73
   macro avg       0.88      0.99      0.92        73
weighted avg       0.99      0.99      0.99        73


=== Top Predicted Teams ===
                      School  Pred_Prob  Actual
118            Illinois NCAA      0.390       1
126          Iowa State NCAA      0.375       1
175            Michigan NCAA      0.350     

In [50]:
print(sorted(y_prob))

[np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.005), np.float64(0.01), np.float6

In [51]:
print(df_cls["Sweet16_or_better"].value_counts())

Sweet16_or_better
0    349
1     16
Name: count, dtype: int64


In [52]:
importance_cls1 = pd.DataFrame({
    "Feature": features_cls1,
    "Importance": rf_cls1.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_cls1)

NameError: name 'features_cls1' is not defined

模型2

In [ ]:
# ===== FINAL CLASSIFICATION MODEL 2 =====

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. 定义目标（Sweet 16及以上）
df2_cls["Sweet16_or_better"] = (df2_cls["NCAA_Result"] >= 3).astype(int)

# 2. 第二模型变量
features_cls2 = [
    "3P_pct",
    "2P_pct",
    "Opp_3P_pct",
    "Opp_2P_pct",
    "ORB_pct",
    "DRB_pct",
    "TOV_pct",
    "STL_pct",
    "Pace",
    "SOS"
]

# 3. 构建数据
data_cls2 = df2_cls[["School"] + features_cls2 + ["Sweet16_or_better"]].dropna()

# 4. 自变量、因变量、学校名
X_cls2 = data_cls2[features_cls2]
y_cls2 = data_cls2["Sweet16_or_better"]
schools = data_cls2["School"]

# 5. 划分训练集和测试集
X_train, X_test, y_train, y_test, school_train, school_test = train_test_split(
    X_cls2,
    y_cls2,
    schools,
    test_size=0.2,
    random_state=42,
    stratify=y_cls2
)

# 6. 训练模型（推荐 Logistic Regression）
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train, y_train)

# 7. 默认预测（threshold = 0.5）
y_pred_default = log_model.predict(X_test)

print("=== Default Prediction ===")
print("Accuracy:", accuracy_score(y_test, y_pred_default))
print(classification_report(y_test, y_pred_default, zero_division=0))

# 8. 预测概率
y_prob = log_model.predict_proba(X_test)[:, 1]

# 9. 查看不同 threshold 的效果
print("\n=== Threshold Comparison ===")
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
    y_pred_temp = (y_prob >= t).astype(int)
    print(f"\nThreshold = {t}")
    print("Accuracy:", accuracy_score(y_test, y_pred_temp))
    print(classification_report(y_test, y_pred_temp, zero_division=0))

# 10. 选择一个最终 threshold
threshold = 0.3
y_pred_adj = (y_prob >= threshold).astype(int)

print("\n=== Adjusted Prediction ===")
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_adj))
print(classification_report(y_test, y_pred_adj, zero_division=0))

# 11. 输出带学校名字的结果
result_df = X_test.copy()
result_df["School"] = school_test.values
result_df["Actual"] = y_test.values
result_df["Pred_Prob"] = y_prob
result_df["Pred_Class"] = y_pred_adj
result_df["NCAA_Result"] = df_cls.loc[X_test.index, "NCAA_Result"].values

print("\n=== Top Teams by Probability ===")
print(
    result_df.sort_values(by="Pred_Prob", ascending=False)[
        ["School", "Pred_Prob", "Pred_Class", "Actual", "NCAA_Result"]
    ].head(10)
)

print("\n=== Teams Predicted to Reach Sweet16 ===")
print(
    result_df[result_df["Pred_Class"] == 1]
    .sort_values(by="Pred_Prob", ascending=False)[
        ["School", "Pred_Prob", "Pred_Class", "Actual", "NCAA_Result"]
    ]
)

# 12. 查看真实进入 Sweet16 的球队
print("\n=== Actual Sweet16 Teams in Test Set ===")
print(
    result_df[result_df["Actual"] == 1]
    .sort_values(by="Pred_Prob", ascending=False)[
        ["School", "Pred_Prob", "Pred_Class", "Actual", "NCAA_Result"]
    ]
)

# 13. 查看模型系数（解释哪些变量更重要）
coef_df = pd.DataFrame({
    "Feature": features_cls2,
    "Coefficient": log_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print("\n=== Feature Coefficients ===")
print(coef_df)

# 14. 统计命中情况
predicted_sweet16 = set(result_df[result_df["Pred_Class"] == 1]["School"])
actual_sweet16 = set(result_df[result_df["Actual"] == 1]["School"])

correct_predictions = predicted_sweet16.intersection(actual_sweet16)

print("\n=== Prediction Summary ===")
print("Predicted Sweet16 teams:", len(predicted_sweet16))
print("Actual Sweet16 teams in test set:", len(actual_sweet16))
print("Correctly predicted:", len(correct_predictions))
print("Correct teams:", correct_predictions)

=== Default Prediction ===
Accuracy: 0.9230769230769231
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.67      0.80         3

    accuracy                           0.92        13
   macro avg       0.95      0.83      0.88        13
weighted avg       0.93      0.92      0.92        13


=== Threshold Comparison ===

Threshold = 0.2
Accuracy: 0.8461538461538461
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        10
           1       0.60      1.00      0.75         3

    accuracy                           0.85        13
   macro avg       0.80      0.90      0.82        13
weighted avg       0.91      0.85      0.86        13


Threshold = 0.25
Accuracy: 0.9230769230769231
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       0.75      1.00      0.86         3

   

In [54]:
# ===== FINAL CLASSIFICATION MODEL 2 (Top16 Method) =====

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. 定义目标（Sweet 16及以上）
df2_cls["Sweet16_or_better"] = (df2_cls["NCAA_Result"] >= 3).astype(int)

# 2. 第二模型变量
features_cls2 = [
    "3P_pct",
    "2P_pct",
    "Opp_3P_pct",
    "Opp_2P_pct",
    "ORB_pct",
    "DRB_pct",
    "TOV_pct",
    "STL_pct",
    "Pace",
    "SOS"
]

# 3. 构建数据
data_cls2 = df2_cls[["School"] + features_cls2 + ["Sweet16_or_better", "NCAA_Result"]].dropna()

# 4. 自变量、因变量、学校名、真实轮次
X_cls2 = data_cls2[features_cls2]
y_cls2 = data_cls2["Sweet16_or_better"]
schools = data_cls2["School"]
ncaa_values = data_cls2["NCAA_Result"]

# 5. 划分训练集和测试集
X_train, X_test, y_train, y_test, school_train, school_test, ncaa_train, ncaa_test = train_test_split(
    X_cls2,
    y_cls2,
    schools,
    ncaa_values,
    test_size=0.2,
    random_state=42,
    stratify=y_cls2
)

# 6. 训练模型
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train, y_train)

# 7. 默认预测（保留做参考）
y_pred_default = log_model.predict(X_test)

print("=== Default Prediction ===")
print("Accuracy:", accuracy_score(y_test, y_pred_default))
print(classification_report(y_test, y_pred_default, zero_division=0))

# 8. 预测概率
y_prob = log_model.predict_proba(X_test)[:, 1]

# 9. 构建结果表
result_df = X_test.copy()
result_df["School"] = school_test.values
result_df["Actual"] = y_test.values
result_df["Pred_Prob"] = y_prob
result_df["NCAA_Result"] = ncaa_test.values

# 10. Top16 方法（核心）
top16 = result_df.sort_values(by="Pred_Prob", ascending=False).head(16)

actual_sweet16 = set(result_df[result_df["Actual"] == 1]["School"])
correct = top16["School"].isin(actual_sweet16).sum()

print("\n=== Top16 Prediction ===")
print(
    top16[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

print("\n命中:", correct, "/16")

# 11. 查看测试集中真实进入 Sweet16 的球队
print("\n=== Actual Sweet16 Teams in Test Set ===")
print(
    result_df[result_df["Actual"] == 1]
    .sort_values(by="Pred_Prob", ascending=False)[
        ["School", "Pred_Prob", "Actual", "NCAA_Result"]
    ]
)

# 12. 查看模型系数
coef_df = pd.DataFrame({
    "Feature": features_cls2,
    "Coefficient": log_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print("\n=== Feature Coefficients ===")
print(coef_df)

# 13. 额外：看Top16里哪些预测对了
correct_teams = top16[top16["School"].isin(actual_sweet16)]

print("\n=== Correctly Predicted Teams in Top16 ===")
print(
    correct_teams[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

# 14. 额外：看Top16里哪些预测错了
wrong_teams = top16[~top16["School"].isin(actual_sweet16)]

print("\n=== Wrong Teams in Top16 ===")
print(
    wrong_teams[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

=== Default Prediction ===
Accuracy: 0.9726027397260274
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        70
           1       0.60      1.00      0.75         3

    accuracy                           0.97        73
   macro avg       0.80      0.99      0.87        73
weighted avg       0.98      0.97      0.98        73


=== Top16 Prediction ===
                      School  Pred_Prob  Actual  NCAA_Result
118            Illinois NCAA   0.997274       1            5
175            Michigan NCAA   0.980824       1            7
126          Iowa State NCAA   0.975232       1            3
16                    Auburn   0.965674       0            0
51              Clemson NCAA   0.888022       0            1
346               Washington   0.375663       0            0
180              Mississippi   0.193313       0            0
324                 UCF NCAA   0.076512       0            1
56                  Colorado   0.066985   

In [ ]:
# ===== FINAL CLASSIFICATION MODEL 2 (Top16 Method) =====

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. 定义目标（Sweet 16及以上）
df2_cls["Sweet16_or_better"] = (df2_cls["NCAA_Result"] >= 3).astype(int)

# 2. 第二模型变量
features_final = [
    "ORtg",
    "Opp_ORtg",
    "3P_pct",
    "2P_pct",
    "ORB_pct",
    "DRB_pct",
    "TOV_pct",
    "SOS"
]
# 3. 构建数据
data_cls2 = df2_cls[["School"] + features_final + ["Sweet16_or_better", "NCAA_Result"]].dropna()

# 4. 自变量、因变量、学校名、真实轮次
X_cls2 = data_cls2[features_final]
y_cls2 = data_cls2["Sweet16_or_better"]
schools = data_cls2["School"]
ncaa_values = data_cls2["NCAA_Result"]

# 5. 划分训练集和测试集
X_train, X_test, y_train, y_test, school_train, school_test, ncaa_train, ncaa_test = train_test_split(
    X_cls2,
    y_cls2,
    schools,
    ncaa_values,
    test_size=0.2,
    random_state=42,
    stratify=y_cls2
)

# 6. 训练模型
log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

log_model.fit(X_train, y_train)

# 7. 默认预测（保留做参考）
y_pred_default = log_model.predict(X_test)

print("=== Default Prediction ===")
print("Accuracy:", accuracy_score(y_test, y_pred_default))
print(classification_report(y_test, y_pred_default, zero_division=0))

# 8. 预测概率
y_prob = log_model.predict_proba(X_test)[:, 1]

# 9. 构建结果表
result_df = X_test.copy()
result_df["School"] = school_test.values
result_df["Actual"] = y_test.values
result_df["Pred_Prob"] = y_prob
result_df["NCAA_Result"] = ncaa_test.values

# 10. Top16 方法（核心）
top16 = result_df.sort_values(by="Pred_Prob", ascending=False).head(16)

actual_sweet16 = set(result_df[result_df["Actual"] == 1]["School"])
correct = top16["School"].isin(actual_sweet16).sum()

print("\n=== Top16 Prediction ===")
print(
    top16[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

print("\n命中:", correct, "/16")

# 11. 查看测试集中真实进入 Sweet16 的球队
print("\n=== Actual Sweet16 Teams in Test Set ===")
print(
    result_df[result_df["Actual"] == 1]
    .sort_values(by="Pred_Prob", ascending=False)[
        ["School", "Pred_Prob", "Actual", "NCAA_Result"]
    ]
)

# 12. 查看模型系数
coef_df = pd.DataFrame({
    "Feature": features_final,
    "Coefficient": log_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print("\n=== Feature Coefficients ===")
print(coef_df)

# 13. 额外：看Top16里哪些预测对了
correct_teams = top16[top16["School"].isin(actual_sweet16)]

print("\n=== Correctly Predicted Teams in Top16 ===")
print(
    correct_teams[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

# 14. 额外：看Top16里哪些预测错了
wrong_teams = top16[~top16["School"].isin(actual_sweet16)]

print("\n=== Wrong Teams in Top16 ===")
print(
    wrong_teams[["School", "Pred_Prob", "Actual", "NCAA_Result"]]
)

=== Default Prediction ===
Accuracy: 0.7692307692307693
              precision    recall  f1-score   support

           0       0.82      0.90      0.86        10
           1       0.50      0.33      0.40         3

    accuracy                           0.77        13
   macro avg       0.66      0.62      0.63        13
weighted avg       0.74      0.77      0.75        13


=== Top16 Prediction ===
                   School     Pred_Prob  Actual  NCAA_Result
126       Iowa State NCAA  8.532692e-01       1            3
337       Vanderbilt NCAA  7.070876e-01       0            2
103          Gonzaga NCAA  4.351476e-01       0            2
290  St. John's (NY) NCAA  3.908091e-01       1            3
194         Nebraska NCAA  2.451768e-01       1            3
207   North Carolina NCAA  2.349031e-01       0            1
174       Miami (OH) NCAA  1.707390e-04       0            1
168          McNeese NCAA  1.232057e-04       0            1
111          Hofstra NCAA  2.379972e-05   

In [ ]:
print(sorted(y_prob))

[np.float64(2.1628887985640203e-18), np.float64(2.2135827161043243e-18), np.float64(5.068882821322694e-18), np.float64(1.297972051438108e-16), np.float64(2.3430658372195193e-16), np.float64(6.095588719715816e-16), np.float64(2.170991211491273e-15), np.float64(2.5503479863394317e-14), np.float64(6.149548494574826e-14), np.float64(9.961779180445826e-14), np.float64(3.303550039575402e-13), np.float64(3.719624741052546e-13), np.float64(3.7331757488001866e-13), np.float64(4.690982023708185e-13), np.float64(5.238582050681207e-13), np.float64(1.0907419659641914e-12), np.float64(1.578642248930063e-12), np.float64(2.9958082653028346e-12), np.float64(3.18429258935352e-12), np.float64(3.238259932628145e-12), np.float64(3.57767586827413e-12), np.float64(6.123954699613366e-12), np.float64(1.421301890898488e-11), np.float64(1.6147583216191732e-11), np.float64(1.9459910264539727e-11), np.float64(2.074123666229664e-11), np.float64(2.2247819307971545e-11), np.float64(2.917669784475043e-11), np.float64(

In [ ]:
importance_cls2 = pd.DataFrame({
    "Feature": features_cls2,
    "Importance": rf_cls2.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_cls2)

      Feature  Importance
9         SOS    0.197627
4     ORB_pct    0.144448
3  Opp_2P_pct    0.127895
7     STL_pct    0.123059
6     TOV_pct    0.093783
2  Opp_3P_pct    0.081996
1      2P_pct    0.076794
8        Pace    0.062278
5     DRB_pct    0.050966
0      3P_pct    0.041155


In [ ]:
print(df2_cls["NCAA_Result"].value_counts().sort_index())

NCAA_Result
0    301
1     32
2     16
3      8
4      4
5      2
6      1
7      1
Name: count, dtype: int64
